# OC14 — DPO on the SFT (Base) model

Direct Preference Optimization on top of the SFT LoRA adapter. **Ordering invariant:** DPO runs on the
SFT model **with the adapter still attached** (`ref_model=None` recovers the reference by disabling the
adapter); the adapter is **merged into the base weights exactly once, after DPO** (the full run).
The SFT adapter is read from the SFT kernel's output (kernel source). **First run: `SMOKE = True`**
(~8 steps). Reads `WANDB_API_KEY`/`HF_TOKEN` from Kaggle Secrets if present.

In [ ]:
# Official Unsloth Kaggle install (verbatim, June 2026): install torch from the cu128
# index FIRST so its wheels carry the T4/sm_75 CUDA kernels. Plain `pip install unsloth`
# let pip resolve a torch build lacking them -> 'CUDA: no kernel image' at model load.
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import subprocess, sys
with open('/kaggle/working/requirements-train.lock.txt', 'w') as fh:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=fh)
print('lockfile written ->', '/kaggle/working/requirements-train.lock.txt')

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _us = UserSecretsClient()
    for _k in ('WANDB_API_KEY', 'HF_TOKEN'):
        try:
            os.environ[_k] = _us.get_secret(_k)
        except Exception:
            pass
except Exception:
    pass
REPORT_TO = 'wandb' if os.environ.get('WANDB_API_KEY') else 'none'
HF_TOKEN = os.environ.get('HF_TOKEN')
print('W&B:', REPORT_TO, '| HF push:', bool(HF_TOKEN))

In [ ]:
import glob
SMOKE = False              # full run (1 epoch + merge once). Set True for an ~8-step validation.
SEED = 3407
_ad = glob.glob('/kaggle/input/**/sft_adapter/adapter_config.json', recursive=True)
assert _ad, 'SFT adapter not found — attach the SFT kernel as a kernel source'
SFT_ADAPTER_DIR = os.path.dirname(_ad[0]); print('SFT_ADAPTER_DIR =', SFT_ADAPTER_DIR)
_dd = glob.glob('/kaggle/input/**/dpo_train.jsonl', recursive=True)
assert _dd, 'dpo_train.jsonl not found'
DATA_DIR = os.path.dirname(_dd[0]); print('DATA_DIR =', DATA_DIR)
OUT_MERGED = '/kaggle/working/dpo_merged_16bit'
HF_REPO = 'ghislaindelabie/oc14-qwen3-1.7b-base-sft-dpo'

In [ ]:
from unsloth import PatchDPOTrainer
PatchDPOTrainer()  # must precede DPOTrainer creation
from unsloth import FastLanguageModel
# Continue from the SFT adapter (do NOT add fresh LoRA — that would discard SFT).
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_DIR, max_seq_length=2048, load_in_4bit=True)
print('eos:', tokenizer.eos_token, '| chat_template set:', tokenizer.chat_template is not None)

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files={
    'train': f'{DATA_DIR}/dpo_train.jsonl', 'val': f'{DATA_DIR}/dpo_val.jsonl'})
keep = {'prompt', 'chosen', 'rejected'}
ds = ds.remove_columns([c for c in ds['train'].column_names if c not in keep])
print(ds)

In [ ]:
from trl import DPOConfig, DPOTrainer
# trl 0.22.2: beta + max_length + max_prompt_length live in DPOConfig; use processing_class=.
cfg = DPOConfig(
    beta=0.1, max_length=2048, max_prompt_length=1024,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    warmup_ratio=0.1, num_train_epochs=1, max_steps=(8 if SMOKE else -1),
    learning_rate=5e-6, logging_steps=5, save_steps=50, save_total_limit=2,
    optim='adamw_8bit', weight_decay=0.0, lr_scheduler_type='linear',
    eval_strategy='steps', eval_steps=50, per_device_eval_batch_size=2,
    seed=SEED, output_dir='/kaggle/working/dpo_out', report_to=REPORT_TO,
    run_name='oc14-dpo-qwen3-base')
trainer = DPOTrainer(model=model, ref_model=None, args=cfg,
                     train_dataset=ds['train'], eval_dataset=ds['val'],
                     processing_class=tokenizer)
import time
_t = time.time(); stats = trainer.train(); print('dpo seconds:', round(time.time() - _t, 1))
print(getattr(stats, 'metrics', stats))

In [ ]:
model.save_pretrained('/kaggle/working/dpo_adapter')
tokenizer.save_pretrained('/kaggle/working/dpo_adapter')
print('saved DPO adapter')
if not SMOKE:
    # Merge ONCE, after DPO -> ordinary 16-bit weights for vLLM. Assert it actually wrote files.
    model.save_pretrained_merged(OUT_MERGED, tokenizer, save_method='merged_16bit')
    files = os.listdir(OUT_MERGED); print('merged files:', files)
    assert any(f.endswith('.safetensors') for f in files), 'merge wrote no weights!'
    if HF_TOKEN:
        model.push_to_hub_merged(HF_REPO, tokenizer, save_method='merged_16bit', token=HF_TOKEN)
        print('pushed merged ->', HF_REPO)

In [ ]:
FastLanguageModel.for_inference(model)
IM_END = tokenizer.convert_tokens_to_ids('<|im_end|>')
sys_msg = ds['train'][0]['prompt'][0]['content']  # the trained system prompt
msgs = [{'role': 'system', 'content': sys_msg},
        {'role': 'user', 'content': 'Un patient de 60 ans a une douleur thoracique aiguë avec sueurs depuis 20 min.'}]
ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
out = model.generate(input_ids=ids, max_new_tokens=200, do_sample=True, temperature=0.3,
                     top_p=0.9, repetition_penalty=1.1, eos_token_id=[IM_END, tokenizer.eos_token_id])
print(tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True))